# Baseline Multi-Label Recommendation Model Training

## Objective

This notebook trains and evaluates baseline machine-learning models for the
context-aware offline resource recommendation component.

The input dataset contains real Knuckles trail, terrain, emergency-access,
communication-infrastructure, and historical weather-context features.

The prediction task is formulated as multi-label classification because a
single hiking context may require multiple offline resources.

The six contextual recommendation targets are:

1. Hypothermia guidance
2. Heat-illness guidance
3. Heavy-rain guidance
4. Strong-wind guidance
5. Water-safety guidance
6. Remote-area emergency guidance

Mandatory safety resources such as offline maps, general first-aid guidance,
and emergency contact information remain outside the ML prediction layer.

The initial baseline models will provide reference performance for later model
comparison and mobile deployment decisions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    accuracy_score,
    classification_report
)

current = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [current, *current.parents]:
    if (
        (candidate / "research").is_dir()
        and (candidate / "pubspec.yaml").is_file()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from: {current}"
    )

RESEARCH_DIR = PROJECT_ROOT / "research"
DATASET_DIR = RESEARCH_DIR / "dataset"
PROCESSED_DIR = DATASET_DIR / "processed"
LABEL_DIR = DATASET_DIR / "labels"
RESULTS_DIR = RESEARCH_DIR / "results"
MODELS_DIR = RESEARCH_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILE = (
    PROCESSED_DIR
    / "features"
    / "knuckles_trail_context_weather_features.csv"
)

LABEL_FILE = (
    LABEL_DIR
    / "knuckles_offline_resource_labels.csv"
)

print("PROJECT PATHS")
print("=" * 60)
print("Project root:", PROJECT_ROOT)
print("Feature dataset exists:", FEATURE_FILE.exists())
print("Label dataset exists:", LABEL_FILE.exists())

PROJECT PATHS
Project root: /Users/mirshad/Desktop/Decentralized-Hiking-AI
Feature dataset exists: True
Label dataset exists: True


In [2]:
features = pd.read_csv(FEATURE_FILE)
labels = pd.read_csv(LABEL_FILE)

print("DATASET LOADING")
print("=" * 60)

print("Feature shape:", features.shape)
print("Label shape:", labels.shape)

print("\nFeature duplicate IDs:")
print(features["id"].duplicated().sum())

print("\nLabel duplicate IDs:")
print(labels["id"].duplicated().sum())

print("\nFeature missing values:")
print(features.isna().sum().sum())

print("\nLabel missing values:")
print(labels.isna().sum().sum())

DATASET LOADING
Feature shape: (557, 49)
Label shape: (557, 8)

Feature duplicate IDs:
0

Label duplicate IDs:
0

Feature missing values:
0

Label missing values:
0


In [3]:
TARGET_COLUMNS = [
    "hypothermia_guidance",
    "heat_illness_guidance",
    "heavy_rain_guidance",
    "strong_wind_guidance",
    "water_safety_guidance",
    "remote_area_emergency_guidance"
]

missing_targets = [
    column
    for column in TARGET_COLUMNS
    if column not in labels.columns
]

print("TARGET VALIDATION")
print("=" * 60)

print("Number of targets:", len(TARGET_COLUMNS))
print("Missing targets:", missing_targets)

for column in TARGET_COLUMNS:
    if column in labels.columns:
        print(
            f"{column}:",
            labels[column].value_counts().sort_index().to_dict()
        )

TARGET VALIDATION
Number of targets: 6
Missing targets: []
hypothermia_guidance: {0: 434, 1: 123}
heat_illness_guidance: {0: 274, 1: 283}
heavy_rain_guidance: {0: 300, 1: 257}
strong_wind_guidance: {0: 351, 1: 206}
water_safety_guidance: {0: 417, 1: 140}
remote_area_emergency_guidance: {0: 310, 1: 247}


In [4]:
model_data = features.merge(
    labels[
        ["id"] + TARGET_COLUMNS
    ],
    on="id",
    how="inner",
    validate="one_to_one"
)

print("MODEL DATASET")
print("=" * 60)

print("Feature rows:", len(features))
print("Label rows:", len(labels))
print("Merged rows:", len(model_data))

print(
    "Duplicate IDs:",
    model_data["id"].duplicated().sum()
)

print(
    "Missing values:",
    model_data.isna().sum().sum()
)

print(
    "Expected 557 trails:",
    len(model_data) == 557
)

display(model_data.head())

MODEL DATASET
Feature rows: 557
Label rows: 557
Merged rows: 557
Duplicate IDs: 0
Missing values: 0
Expected 557 trails: True


,id,highway,length_km,representative_latitude,representative_longitude,minimum_elevation_m,maximum_elevation_m,elevation_range_m,elevation_gain_m,elevation_loss_m,...,hist_apparent_temperature_p05_c,hist_apparent_temperature_p95_c,hist_rain_hour_fraction,hist_rain_hour_percent,hypothermia_guidance,heat_illness_guidance,heavy_rain_guidance,strong_wind_guidance,water_safety_guidance,remote_area_emergency_guidance
0,way/100146710,footway,1.654853,7.420997,80.712054,1275.355835,1329.643188,54.287354,137.471680,83.390137,...,18.9,27.5,0.310948,31.094834,1,0,0,1,0,0
1,way/100146711,track,4.966901,7.418119,80.703434,1024.330322,1281.052856,256.722534,485.684814,287.749878,...,18.9,27.5,0.310948,31.094834,1,0,0,1,0,0
2,way/100148512,track,0.995269,7.429983,80.711814,1332.734375,1420.415405,87.681030,116.777222,29.096191,...,18.9,27.5,0.310948,31.094834,1,0,0,1,0,0
3,way/100398799,footway,0.073670,7.380225,80.680681,784.558533,789.260803,4.702271,3.243958,4.702271,...,23.5,32.4,0.342849,34.284867,0,1,1,0,0,1
4,way/100398800,steps,0.195638,7.383670,80.678803,696.232178,753.476257,57.244080,57.244080,0.000000,...,23.5,32.4,0.342849,34.284867,0,1,1,0,0,1


## Feature Selection and Target-Leakage Control

The recommendation targets in this study are rule-derived labels rather than
independently observed expert decisions or real-world hiking outcomes.

Several labels were constructed directly from environmental and remoteness
features. Consequently, training a model using exactly the same variables that
defined a target can result in target leakage: the model may simply reproduce
the labeling rule.

To reduce this problem, direct rule-defining variables are excluded from the
initial baseline feature set where practical.

The resulting evaluation should still be interpreted as an assessment of how
well machine-learning models approximate the constructed recommendation
policy, rather than independent validation of real-world safety effectiveness.

Future validation should use independent expert review and/or real-world
recommendation outcomes.

In [5]:
DIRECT_LABEL_RULE_FEATURES = [
    # Hypothermia rule
    "hist_temperature_p05_c",
    "elevation_gain_m",
    "maximum_elevation_m",

    # Heat rule
    "hist_apparent_temperature_p95_c",

    # Rain rule
    "hist_precipitation_p95_mm",
    "hist_rain_hour_percent",

    # Wind rule
    "hist_wind_gust_p95_kmh",

    # Water rule
    "nearest_drinking_water_km",

    # Remote-area rule
    "nearest_hospital_km",
    "nearest_ranger_station_km",
    "nearest_communication_structure_km"
]

print("DIRECT LABEL-RULE FEATURES")
print("=" * 60)

for column in DIRECT_LABEL_RULE_FEATURES:
    print("-", column)

print(
    "\nNumber of direct rule features:",
    len(DIRECT_LABEL_RULE_FEATURES)
)

DIRECT LABEL-RULE FEATURES
- hist_temperature_p05_c
- elevation_gain_m
- maximum_elevation_m
- hist_apparent_temperature_p95_c
- hist_precipitation_p95_mm
- hist_rain_hour_percent
- hist_wind_gust_p95_kmh
- nearest_drinking_water_km
- nearest_hospital_km
- nearest_ranger_station_km
- nearest_communication_structure_km

Number of direct rule features: 11


In [6]:
EXCLUDED_MODEL_COLUMNS = (
    ["id"]
    + TARGET_COLUMNS
    + DIRECT_LABEL_RULE_FEATURES
    + [
        "point_id",
        "hist_rain_hour_fraction"
    ]
)

MODEL_FEATURE_COLUMNS = [
    column
    for column in model_data.columns
    if column not in EXCLUDED_MODEL_COLUMNS
]

print("BASELINE MODEL FEATURES")
print("=" * 60)

print(
    "Number of selected features:",
    len(MODEL_FEATURE_COLUMNS)
)

print("\nSelected features:")

for column in MODEL_FEATURE_COLUMNS:
    print("-", column)

BASELINE MODEL FEATURES
Number of selected features: 35

Selected features:
- highway
- length_km
- representative_latitude
- representative_longitude
- minimum_elevation_m
- elevation_range_m
- elevation_loss_m
- mean_slope_deg
- median_slope_deg
- max_slope_deg
- slope_std_deg
- terrain_sample_count
- nearest_police_km
- nearest_clinic_km
- nearest_doctors_km
- nearest_shelter_km
- weather_station_distance_km
- hist_temperature_mean_c
- hist_temperature_min_c
- hist_temperature_max_c
- hist_apparent_temperature_mean_c
- hist_apparent_temperature_min_c
- hist_apparent_temperature_max_c
- hist_relative_humidity_mean_pct
- hist_precipitation_mean_mm
- hist_precipitation_max_mm
- hist_wind_speed_mean_kmh
- hist_wind_speed_max_kmh
- hist_wind_gust_mean_kmh
- hist_wind_gust_max_kmh
- hist_cloud_cover_mean_pct
- hist_surface_pressure_mean_hpa
- hist_wind_speed_p95_kmh
- hist_temperature_p95_c
- hist_apparent_temperature_p05_c


In [7]:
X = model_data[
    MODEL_FEATURE_COLUMNS
].copy()

y = model_data[
    TARGET_COLUMNS
].copy()

categorical_features = (
    X.select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

numeric_features = (
    X.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)

print("MODEL MATRICES")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

print(
    "\nCategorical features:",
    categorical_features
)

print(
    "\nNumber of numeric features:",
    len(numeric_features)
)

print(
    "\nUnclassified features:",
    set(X.columns)
    - set(categorical_features)
    - set(numeric_features)
)

MODEL MATRICES
X shape: (557, 35)
y shape: (557, 6)

Categorical features: ['highway']

Number of numeric features: 34

Unclassified features: set()


In [8]:
leaked_rule_features = sorted(
    set(MODEL_FEATURE_COLUMNS)
    .intersection(
        DIRECT_LABEL_RULE_FEATURES
    )
)

target_leakage = sorted(
    set(MODEL_FEATURE_COLUMNS)
    .intersection(
        TARGET_COLUMNS
    )
)

print("LEAKAGE CHECK")
print("=" * 60)

print(
    "Direct rule features remaining:",
    leaked_rule_features
)

print(
    "Target columns remaining:",
    target_leakage
)

LEAKAGE CHECK
Direct rule features remaining: []
Target columns remaining: []


In [9]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )
)

print("TRAIN / TEST SPLIT")
print("=" * 60)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target prevalence (%):")

print(
    (
        y_train.mean() * 100
    ).round(2)
)

print("\nTest target prevalence (%):")

print(
    (
        y_test.mean() * 100
    ).round(2)
)

TRAIN / TEST SPLIT
X_train: (445, 35)
X_test : (112, 35)
y_train: (445, 6)
y_test : (112, 6)

Training target prevalence (%):
hypothermia_guidance              22.70
heat_illness_guidance             50.34
heavy_rain_guidance               45.39
strong_wind_guidance              37.75
water_safety_guidance             24.04
remote_area_emergency_guidance    43.82
dtype: float64

Test target prevalence (%):
hypothermia_guidance              19.64
heat_illness_guidance             52.68
heavy_rain_guidance               49.11
strong_wind_guidance              33.93
water_safety_guidance             29.46
remote_area_emergency_guidance    46.43
dtype: float64


In [10]:
split_balance = pd.DataFrame({
    "full_percent":
        y.mean() * 100,

    "train_percent":
        y_train.mean() * 100,

    "test_percent":
        y_test.mean() * 100
})

split_balance[
    "train_difference"
] = (
    split_balance["train_percent"]
    - split_balance["full_percent"]
)

split_balance[
    "test_difference"
] = (
    split_balance["test_percent"]
    - split_balance["full_percent"]
)

print("TRAIN / TEST LABEL BALANCE")
print("=" * 60)

display(
    split_balance.round(2)
)

TRAIN / TEST LABEL BALANCE


,full_percent,train_percent,test_percent,train_difference,test_difference
hypothermia_guidance,22.08,22.70,19.64,0.61,-2.44
heat_illness_guidance,50.81,50.34,52.68,-0.47,1.87
heavy_rain_guidance,46.14,45.39,49.11,-0.75,2.97
strong_wind_guidance,36.98,37.75,33.93,0.77,-3.06
water_safety_guidance,25.13,24.04,29.46,-1.09,4.33
remote_area_emergency_guidance,44.34,43.82,46.43,-0.52,2.08


## Baseline Models

Two baseline algorithms are evaluated:

1. **Logistic Regression**
   - Provides a simple and interpretable linear baseline.
   - Numerical features are standardized.
   - Categorical features are one-hot encoded.
   - A separate binary classifier is trained for each recommendation target.

2. **Random Forest**
   - Provides a nonlinear tree-based baseline.
   - Can capture interactions between trail, terrain, remoteness, and
     environmental variables.
   - Uses the same multi-output classification formulation.

The models are evaluated using precision, recall, F1-score, Hamming Loss,
and Exact-Match Accuracy.

Because the targets are rule-derived, these results measure approximation of
the constructed recommendation policy and must not be interpreted as
independent validation of hiking-safety effectiveness.

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

print("PREPROCESSING PIPELINE")
print("=" * 60)

print(
    "Numeric features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    categorical_features
)

print("\nPreprocessor created successfully.")

PREPROCESSING PIPELINE
Numeric features: 34
Categorical features: ['highway']

Preprocessor created successfully.


In [12]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            MultiOutputClassifier(
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=42
                )
            )
        )
    ]
)

print("TRAINING LOGISTIC REGRESSION")
print("=" * 60)

logistic_model.fit(
    X_train,
    y_train
)

logistic_predictions = logistic_model.predict(
    X_test
)

print("Training complete.")

print(
    "Prediction shape:",
    logistic_predictions.shape
)

TRAINING LOGISTIC REGRESSION
Training complete.
Prediction shape: (112, 6)


In [13]:
logistic_precision = precision_score(
    y_test,
    logistic_predictions,
    average="macro",
    zero_division=0
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions,
    average="macro",
    zero_division=0
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions,
    average="macro",
    zero_division=0
)

logistic_hamming = hamming_loss(
    y_test,
    logistic_predictions
)

logistic_exact_match = accuracy_score(
    y_test,
    logistic_predictions
)

print("LOGISTIC REGRESSION RESULTS")
print("=" * 60)

print(
    f"Macro Precision: {logistic_precision:.4f}"
)
print(
    f"Macro Recall: {logistic_recall:.4f}"
)
print(
    f"Macro F1: {logistic_f1:.4f}"
)
print(
    f"Hamming Loss: {logistic_hamming:.4f}"
)
print(
    f"Exact-Match Accuracy: {logistic_exact_match:.4f}"
)

LOGISTIC REGRESSION RESULTS
Macro Precision: 0.9337
Macro Recall: 0.9562
Macro F1: 0.9444
Hamming Loss: 0.0357
Exact-Match Accuracy: 0.8036


In [14]:
logistic_report = classification_report(
    y_test,
    logistic_predictions,
    target_names=TARGET_COLUMNS,
    output_dict=True,
    zero_division=0
)

logistic_per_label = pd.DataFrame(
    logistic_report
).T.loc[
    TARGET_COLUMNS,
    [
        "precision",
        "recall",
        "f1-score",
        "support"
    ]
]

print("LOGISTIC REGRESSION — PER LABEL")
print("=" * 60)

display(
    logistic_per_label.round(4)
)

LOGISTIC REGRESSION — PER LABEL


,precision,recall,f1-score,support
hypothermia_guidance,0.8750,0.9545,0.9130,22.0
heat_illness_guidance,1.0000,1.0000,1.0000,59.0
heavy_rain_guidance,1.0000,1.0000,1.0000,55.0
strong_wind_guidance,1.0000,1.0000,1.0000,38.0
water_safety_guidance,0.8056,0.8788,0.8406,33.0
remote_area_emergency_guidance,0.9216,0.9038,0.9126,52.0


In [15]:
random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            MultiOutputClassifier(
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1
                )
            )
        )
    ]
)

print("TRAINING RANDOM FOREST")
print("=" * 60)

random_forest_model.fit(
    X_train,
    y_train
)

rf_predictions = random_forest_model.predict(
    X_test
)

print("Training complete.")

print(
    "Prediction shape:",
    rf_predictions.shape
)

TRAINING RANDOM FOREST
Training complete.
Prediction shape: (112, 6)


In [16]:
rf_precision = precision_score(
    y_test,
    rf_predictions,
    average="macro",
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_predictions,
    average="macro",
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_predictions,
    average="macro",
    zero_division=0
)

rf_hamming = hamming_loss(
    y_test,
    rf_predictions
)

rf_exact_match = accuracy_score(
    y_test,
    rf_predictions
)

print("RANDOM FOREST RESULTS")
print("=" * 60)

print(
    f"Macro Precision: {rf_precision:.4f}"
)
print(
    f"Macro Recall: {rf_recall:.4f}"
)
print(
    f"Macro F1: {rf_f1:.4f}"
)
print(
    f"Hamming Loss: {rf_hamming:.4f}"
)
print(
    f"Exact-Match Accuracy: {rf_exact_match:.4f}"
)

RANDOM FOREST RESULTS
Macro Precision: 0.9826
Macro Recall: 0.9443
Macro F1: 0.9613
Hamming Loss: 0.0238
Exact-Match Accuracy: 0.8571


In [17]:
rf_report = classification_report(
    y_test,
    rf_predictions,
    target_names=TARGET_COLUMNS,
    output_dict=True,
    zero_division=0
)

rf_per_label = pd.DataFrame(
    rf_report
).T.loc[
    TARGET_COLUMNS,
    [
        "precision",
        "recall",
        "f1-score",
        "support"
    ]
]

print("RANDOM FOREST — PER LABEL")
print("=" * 60)

display(
    rf_per_label.round(4)
)

RANDOM FOREST — PER LABEL


,precision,recall,f1-score,support
hypothermia_guidance,0.9545,0.9545,0.9545,22.0
heat_illness_guidance,1.0000,1.0000,1.0000,59.0
heavy_rain_guidance,1.0000,1.0000,1.0000,55.0
strong_wind_guidance,1.0000,1.0000,1.0000,38.0
water_safety_guidance,1.0000,0.7879,0.8814,33.0
remote_area_emergency_guidance,0.9412,0.9231,0.9320,52.0


In [18]:
model_comparison = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Random Forest"
        ],
        "Macro Precision": [
            logistic_precision,
            rf_precision
        ],
        "Macro Recall": [
            logistic_recall,
            rf_recall
        ],
        "Macro F1": [
            logistic_f1,
            rf_f1
        ],
        "Hamming Loss": [
            logistic_hamming,
            rf_hamming
        ],
        "Exact Match Accuracy": [
            logistic_exact_match,
            rf_exact_match
        ]
    }
)

model_comparison = (
    model_comparison
    .sort_values(
        "Macro F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("BASELINE MODEL COMPARISON")
print("=" * 60)

display(
    model_comparison.round(4)
)

BASELINE MODEL COMPARISON


,Model,Macro Precision,Macro Recall,Macro F1,Hamming Loss,Exact Match Accuracy
0,Random Forest,0.9826,0.9443,0.9613,0.0238,0.8571
1,Logistic Regression,0.9337,0.9562,0.9444,0.0357,0.8036


## Initial Baseline Results

Random Forest achieved the strongest initial random-split performance, with a
macro F1-score of 0.9613 and Hamming Loss of 0.0238. Logistic Regression
achieved a macro F1-score of 0.9444.

Perfect test-set scores were observed for several weather-related recommendation
labels.

These high results must be interpreted cautiously. The recommendation targets
are rule-derived, and multiple remaining predictors are correlated with the
variables used during label construction. In addition, historical weather
context is derived from only five Knuckles reference locations. A random
trail-level split can therefore place trails sharing the same environmental
profile in both training and testing data.

The random-split experiment is retained as an initial policy-approximation
baseline rather than evidence of independent real-world safety accuracy.

A stricter grouped evaluation is performed next to measure generalization when
weather-context groups are held out from training.

In [19]:
weather_groups = features[
    ["id", "point_id"]
].copy()

print("WEATHER GROUPS")
print("=" * 60)

print(
    "Number of weather groups:",
    weather_groups["point_id"].nunique()
)

print("\nTrails per weather group:")

print(
    weather_groups["point_id"]
    .value_counts()
    .sort_index()
)

WEATHER GROUPS
Number of weather groups: 5

Trails per weather group:
point_id
knuckles_0    132
knuckles_1    257
knuckles_2     68
knuckles_3     74
knuckles_4     26
Name: count, dtype: int64


In [20]:
from sklearn.model_selection import LeaveOneGroupOut

groups = model_data["point_id"].copy()

logo = LeaveOneGroupOut()

print("GROUPED EVALUATION SETUP")
print("=" * 60)

print(
    "Number of groups:",
    groups.nunique()
)

print(
    "Number of folds:",
    logo.get_n_splits(
        X,
        y,
        groups
    )
)

print("\nGroup sizes:")

print(
    groups.value_counts()
    .sort_index()
)

GROUPED EVALUATION SETUP
Number of groups: 5
Number of folds: 5

Group sizes:
point_id
knuckles_0    132
knuckles_1    257
knuckles_2     68
knuckles_3     74
knuckles_4     26
Name: count, dtype: int64


In [22]:
from sklearn.base import clone

grouped_logistic_results = []
grouped_logistic_single_class = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y, groups),
    start=1
):
    X_group_train = X.iloc[train_idx]
    X_group_test = X.iloc[test_idx]

    y_group_train = y.iloc[train_idx]
    y_group_test = y.iloc[test_idx]

    test_group = groups.iloc[test_idx].iloc[0]

    fold_predictions = np.zeros(
        (
            len(test_idx),
            len(TARGET_COLUMNS)
        ),
        dtype=int
    )

    for label_index, label in enumerate(
        TARGET_COLUMNS
    ):
        y_label_train = y_group_train[label]

        unique_classes = np.sort(
            y_label_train.unique()
        )

        if len(unique_classes) == 1:

            constant_class = int(
                unique_classes[0]
            )

            fold_predictions[
                :,
                label_index
            ] = constant_class

            grouped_logistic_single_class.append({
                "fold": fold,
                "held_out_group": test_group,
                "label": label,
                "training_class": constant_class
            })

        else:

            label_model = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        clone(preprocessor)
                    ),
                    (
                        "classifier",
                        LogisticRegression(
                            max_iter=2000,
                            class_weight="balanced",
                            random_state=42
                        )
                    )
                ]
            )

            label_model.fit(
                X_group_train,
                y_label_train
            )

            fold_predictions[
                :,
                label_index
            ] = label_model.predict(
                X_group_test
            )

    grouped_logistic_results.append({
        "fold": fold,
        "held_out_group": test_group,
        "test_size": len(test_idx),

        "macro_precision": precision_score(
            y_group_test,
            fold_predictions,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_group_test,
            fold_predictions,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_group_test,
            fold_predictions,
            average="macro",
            zero_division=0
        ),

        "hamming_loss": hamming_loss(
            y_group_test,
            fold_predictions
        ),

        "exact_match_accuracy": accuracy_score(
            y_group_test,
            fold_predictions
        )
    })

grouped_logistic_results = pd.DataFrame(
    grouped_logistic_results
)

grouped_logistic_single_class = pd.DataFrame(
    grouped_logistic_single_class
)

print("GROUPED LOGISTIC REGRESSION")
print("=" * 60)

display(
    grouped_logistic_results.round(4)
)

print("\nSINGLE-CLASS TRAINING CASES")
print("=" * 60)

if grouped_logistic_single_class.empty:
    print("None")
else:
    display(
        grouped_logistic_single_class
    )

GROUPED LOGISTIC REGRESSION


,fold,held_out_group,test_size,macro_precision,macro_recall,macro_f1,hamming_loss,exact_match_accuracy
0,1,knuckles_0,132,0.4724,0.6263,0.5238,0.1414,0.3182
1,2,knuckles_1,257,0.0402,0.1667,0.0648,0.5136,0.0000
2,3,knuckles_2,68,0.0657,0.1667,0.0942,0.2966,0.0000
3,4,knuckles_3,74,0.3844,0.5000,0.4115,0.1216,0.3108
4,5,knuckles_4,26,0.5000,0.3590,0.3778,0.3077,0.0000



SINGLE-CLASS TRAINING CASES


,fold,held_out_group,label,training_class
0,2,knuckles_1,heavy_rain_guidance,0


In [23]:
grouped_rf_results = []
grouped_rf_single_class = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y, groups),
    start=1
):
    X_group_train = X.iloc[train_idx]
    X_group_test = X.iloc[test_idx]

    y_group_train = y.iloc[train_idx]
    y_group_test = y.iloc[test_idx]

    test_group = groups.iloc[test_idx].iloc[0]

    fold_predictions = np.zeros(
        (
            len(test_idx),
            len(TARGET_COLUMNS)
        ),
        dtype=int
    )

    for label_index, label in enumerate(
        TARGET_COLUMNS
    ):
        y_label_train = y_group_train[label]

        unique_classes = np.sort(
            y_label_train.unique()
        )

        if len(unique_classes) == 1:

            constant_class = int(
                unique_classes[0]
            )

            fold_predictions[
                :,
                label_index
            ] = constant_class

            grouped_rf_single_class.append({
                "fold": fold,
                "held_out_group": test_group,
                "label": label,
                "training_class": constant_class
            })

        else:

            label_model = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        clone(preprocessor)
                    ),
                    (
                        "classifier",
                        RandomForestClassifier(
                            n_estimators=300,
                            class_weight="balanced",
                            random_state=42,
                            n_jobs=-1
                        )
                    )
                ]
            )

            label_model.fit(
                X_group_train,
                y_label_train
            )

            fold_predictions[
                :,
                label_index
            ] = label_model.predict(
                X_group_test
            )

    grouped_rf_results.append({
        "fold": fold,
        "held_out_group": test_group,
        "test_size": len(test_idx),

        "macro_precision": precision_score(
            y_group_test,
            fold_predictions,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_group_test,
            fold_predictions,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_group_test,
            fold_predictions,
            average="macro",
            zero_division=0
        ),

        "hamming_loss": hamming_loss(
            y_group_test,
            fold_predictions
        ),

        "exact_match_accuracy": accuracy_score(
            y_group_test,
            fold_predictions
        )
    })

grouped_rf_results = pd.DataFrame(
    grouped_rf_results
)

grouped_rf_single_class = pd.DataFrame(
    grouped_rf_single_class
)

print("GROUPED RANDOM FOREST")
print("=" * 60)

display(
    grouped_rf_results.round(4)
)

print("\nSINGLE-CLASS TRAINING CASES")
print("=" * 60)

if grouped_rf_single_class.empty:
    print("None")
else:
    display(
        grouped_rf_single_class
    )

GROUPED RANDOM FOREST


,fold,held_out_group,test_size,macro_precision,macro_recall,macro_f1,hamming_loss,exact_match_accuracy
0,1,knuckles_0,132,0.5745,0.3597,0.3184,0.2273,0.0606
1,2,knuckles_1,257,0.0404,0.1667,0.0650,0.5123,0.0000
2,3,knuckles_2,68,0.1193,0.1385,0.1263,0.3015,0.0000
3,4,knuckles_3,74,0.5000,0.3326,0.3641,0.0878,0.5405
4,5,knuckles_4,26,0.3333,0.2756,0.2984,0.3910,0.0000



SINGLE-CLASS TRAINING CASES


,fold,held_out_group,label,training_class
0,2,knuckles_1,heavy_rain_guidance,0


In [24]:
grouped_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Mean Macro Precision": [
        grouped_logistic_results[
            "macro_precision"
        ].mean(),

        grouped_rf_results[
            "macro_precision"
        ].mean()
    ],

    "Mean Macro Recall": [
        grouped_logistic_results[
            "macro_recall"
        ].mean(),

        grouped_rf_results[
            "macro_recall"
        ].mean()
    ],

    "Mean Macro F1": [
        grouped_logistic_results[
            "macro_f1"
        ].mean(),

        grouped_rf_results[
            "macro_f1"
        ].mean()
    ],

    "Mean Hamming Loss": [
        grouped_logistic_results[
            "hamming_loss"
        ].mean(),

        grouped_rf_results[
            "hamming_loss"
        ].mean()
    ],

    "Mean Exact Match": [
        grouped_logistic_results[
            "exact_match_accuracy"
        ].mean(),

        grouped_rf_results[
            "exact_match_accuracy"
        ].mean()
    ]
})

print("GROUPED EVALUATION SUMMARY")
print("=" * 60)

display(
    grouped_summary.round(4)
)

GROUPED EVALUATION SUMMARY


,Model,Mean Macro Precision,Mean Macro Recall,Mean Macro F1,Mean Hamming Loss,Mean Exact Match
0,Logistic Regression,0.2925,0.3637,0.2944,0.2762,0.1258
1,Random Forest,0.3135,0.2546,0.2344,0.3040,0.1202


## Grouped Generalization Evaluation

The initial random train/test split produced high performance, with Random
Forest achieving a macro F1-score of 0.9613 and Logistic Regression achieving
0.9444.

However, the stricter Leave-One-Weather-Group-Out evaluation produced
substantially lower performance:

- Logistic Regression mean macro F1: 0.2944
- Random Forest mean macro F1: 0.2344
- Logistic Regression mean Hamming Loss: 0.2762
- Random Forest mean Hamming Loss: 0.3040

This difference demonstrates that the random trail-level split provides an
optimistic estimate of performance because trails associated with the same
regional weather profiles can occur in both the training and test sets.

The grouped evaluation is substantially more challenging because the model
must predict recommendations for a weather-context group that was not observed
during training.

Performance also varies considerably between held-out groups. In particular,
holding out the knuckles_1 group produced very low performance for both
models.

For the heavy_rain_guidance target, removing knuckles_1 resulted in a training
set containing only the negative class. This indicates that the current
rule-derived rain target is strongly associated with a limited number of
weather profiles.

These findings highlight an important limitation of the current dataset:
historical weather context is represented by only five reference locations.

Therefore, the high random-split results are treated as policy-approximation
performance, while the grouped results provide a more conservative indication
of generalization to unseen regional environmental contexts.

Neither evaluation is interpreted as independent validation of real-world
hiking-safety effectiveness.

In [25]:
evaluation_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Random Split Macro F1": [
        logistic_f1,
        rf_f1
    ],

    "Grouped Mean Macro F1": [
        grouped_logistic_results[
            "macro_f1"
        ].mean(),

        grouped_rf_results[
            "macro_f1"
        ].mean()
    ],

    "Random Split Hamming Loss": [
        logistic_hamming,
        rf_hamming
    ],

    "Grouped Mean Hamming Loss": [
        grouped_logistic_results[
            "hamming_loss"
        ].mean(),

        grouped_rf_results[
            "hamming_loss"
        ].mean()
    ]
})

evaluation_comparison[
    "Macro F1 Drop"
] = (
    evaluation_comparison[
        "Random Split Macro F1"
    ]
    -
    evaluation_comparison[
        "Grouped Mean Macro F1"
    ]
)

print("RANDOM VS GROUPED EVALUATION")
print("=" * 60)

display(
    evaluation_comparison.round(4)
)

RANDOM VS GROUPED EVALUATION


,Model,Random Split Macro F1,Grouped Mean Macro F1,Random Split Hamming Loss,Grouped Mean Hamming Loss,Macro F1 Drop
0,Logistic Regression,0.9444,0.2944,0.0357,0.2762,0.6500
1,Random Forest,0.9613,0.2344,0.0238,0.3040,0.7269


In [26]:
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_comparison.to_csv(
    RESULTS_DIR
    / "baseline_random_split_results.csv",
    index=False
)

grouped_logistic_results.to_csv(
    RESULTS_DIR
    / "grouped_logistic_results.csv",
    index=False
)

grouped_rf_results.to_csv(
    RESULTS_DIR
    / "grouped_random_forest_results.csv",
    index=False
)

grouped_summary.to_csv(
    RESULTS_DIR
    / "grouped_evaluation_summary.csv",
    index=False
)

evaluation_comparison.to_csv(
    RESULTS_DIR
    / "random_vs_grouped_evaluation.csv",
    index=False
)

if not grouped_logistic_single_class.empty:
    grouped_logistic_single_class.to_csv(
        RESULTS_DIR
        / "grouped_single_class_cases.csv",
        index=False
    )

print("RESULT FILES SAVED")
print("=" * 60)

for file in sorted(
    RESULTS_DIR.glob("*.csv")
):
    print("-", file.name)

RESULT FILES SAVED
- baseline_random_split_results.csv
- dataset_audit_summary.csv
- grouped_evaluation_summary.csv
- grouped_logistic_results.csv
- grouped_random_forest_results.csv
- grouped_single_class_cases.csv
- random_vs_grouped_evaluation.csv


In [27]:
required_result_files = [
    "baseline_random_split_results.csv",
    "grouped_logistic_results.csv",
    "grouped_random_forest_results.csv",
    "grouped_evaluation_summary.csv",
    "random_vs_grouped_evaluation.csv"
]

print("FINAL VALIDATION")
print("=" * 60)

all_valid = True

for filename in required_result_files:

    path = RESULTS_DIR / filename
    exists = path.exists()

    print(
        f"{filename}:",
        exists
    )

    if not exists:
        all_valid = False

print("\nAll required results saved:", all_valid)

FINAL VALIDATION
baseline_random_split_results.csv: True
grouped_logistic_results.csv: True
grouped_random_forest_results.csv: True
grouped_evaluation_summary.csv: True
random_vs_grouped_evaluation.csv: True

All required results saved: True


## Baseline Conclusion

Both baseline models successfully learned patterns associated with the
constructed offline-resource recommendation policy under a random trail-level
split.

Random Forest produced the strongest random-split result, whereas Logistic
Regression produced the stronger mean macro F1 under the stricter
Leave-One-Weather-Group-Out evaluation.

The large performance reduction under grouped evaluation demonstrates that
generalization across unseen environmental regions remains challenging.

The current prototype therefore establishes a working multi-label
recommendation pipeline but also identifies the limited number of weather
reference locations and rule-derived target construction as important
limitations requiring further validation and data expansion.

Future development should include broader spatial weather coverage,
independent expert review of recommendation targets, and evaluation using
additional real hiking contexts before safety effectiveness claims are made.